In [1]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
MODEL = os.getenv("CHAT_MODEL")

In [3]:
from langchain.tools import tool

In [6]:
@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"

In [4]:
model = init_chat_model(MODEL)

In [7]:
model_with_tools=model.bind_tools([get_weather])

In [14]:
response = model_with_tools.invoke("Hi")
response.to_json()

{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'messages', 'AIMessage'],
 'kwargs': {'content': 'Hi! How can I help you today?',
  'additional_kwargs': {'refusal': None},
  'response_metadata': {'token_usage': {'completion_tokens': 12,
    'prompt_tokens': 126,
    'total_tokens': 138,
    'completion_tokens_details': {'accepted_prediction_tokens': 0,
     'audio_tokens': 0,
     'reasoning_tokens': 0,
     'rejected_prediction_tokens': 0},
    'prompt_tokens_details': {'audio_tokens': 0,
     'cache_write_tokens': None,
     'cached_tokens': 0}},
   'model_provider': 'openai',
   'model_name': 'gpt-5.4-nano-2026-03-17',
   'system_fingerprint': None,
   'id': 'chatcmpl-EBoJ5ZyBR7buYTmucI20aPbT5a5Md',
   'service_tier': 'default',
   'finish_reason': 'stop',
   'logprobs': None},
  'type': 'ai',
  'id': 'lc_run--019ff2b9-da91-7610-8921-1a78ea034d58-0',
  'usage_metadata': {'input_tokens': 126,
   'output_tokens': 12,
   'total_tokens': 138,
   'input_token_details': {

In [ ]:
#    'finish_reason': 'stop',

In [10]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

content='' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 132, 'total_tokens': 149, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5.4-nano-2026-03-17', 'system_fingerprint': None, 'id': 'chatcmpl-EBoHVfpEglhpDCJJngbTqCJCKKmVQ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019ff2b8-5c4b-73c2-a994-354f16a514d9-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'call_8jQVy4SEz82FELNU7dvtu7d9', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 132, 'output_tokens': 17, 'total_tokens': 149, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Tool:

In [ ]:
response.to_json()
# 'finish_reason': 'tool_calls',


{'lc': 1,
 'type': 'constructor',
 'id': ['langchain', 'schema', 'messages', 'AIMessage'],
 'kwargs': {'content': '',
  'additional_kwargs': {'refusal': None},
  'response_metadata': {'token_usage': {'completion_tokens': 17,
    'prompt_tokens': 132,
    'total_tokens': 149,
    'completion_tokens_details': {'accepted_prediction_tokens': 0,
     'audio_tokens': 0,
     'reasoning_tokens': 0,
     'rejected_prediction_tokens': 0},
    'prompt_tokens_details': {'audio_tokens': 0,
     'cache_write_tokens': None,
     'cached_tokens': 0}},
   'model_provider': 'openai',
   'model_name': 'gpt-5.4-nano-2026-03-17',
   'system_fingerprint': None,
   'id': 'chatcmpl-EBoHVfpEglhpDCJJngbTqCJCKKmVQ',
   'service_tier': 'default',
   'finish_reason': 'tool_calls',
   'logprobs': None},
  'type': 'ai',
  'id': 'lc_run--019ff2b8-5c4b-73c2-a994-354f16a514d9-0',
  'tool_calls': [{'name': 'get_weather',
    'args': {'location': 'Boston'},
    'id': 'call_8jQVy4SEz82FELNU7dvtu7d9',
    'type': 'tool_ca

In [15]:
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage

In [ ]:
messages = [
    SystemMessage(content="You are a helpful assistant. Use tools when needed and answer clearly."),
    HumanMessage(content="What's the weather in Boston?") #question
]

In [17]:
#  Safety cap for production; increase if needed.
max_total_tool_calls = 20
tool_calls_made = 0

while True:
    ai_msg = model_with_tools.invoke(messages)
    messages.append(ai_msg)

    # Model is done calling tools -> final answer ready
    if not ai_msg.tool_calls:
        final_response = ai_msg
        break

    for tool_call in ai_msg.tool_calls:
        if tool_call["name"] == "get_weather":
            tool_result = get_weather.invoke(tool_call["args"])
        else:
            tool_result = f"Unknown tool: {tool_call['name']}"

        messages.append(
            ToolMessage(
                content=str(tool_result),
                tool_call_id=tool_call["id"],
                name=tool_call["name"]
            )
        )

        tool_calls_made += 1
        if tool_calls_made >= max_total_tool_calls:
            raise RuntimeError("Max tool calls reached before final model response.")

print(final_response.content)

Right now in **Boston**, it’s **sunny**.
